# DuckGPT — 02. Attention, visualised

> Companion notebook to [`core/03_attention.py`](../core/03_attention.py).

We run the same maths from `core/03_attention.py` inline and plot the
attention matrix for a single head, with and without the causal mask.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
T, d = 8, 16

def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

X = rng.standard_normal((T, d))
Wq, Wk = rng.standard_normal((d, d)), rng.standard_normal((d, d))
Q, K = X @ Wq, X @ Wk
scores = Q @ K.T / math.sqrt(d)
attn_unmasked = softmax(scores)
mask = np.tril(np.ones((T, T), dtype=bool))
attn_causal = softmax(np.where(mask, scores, -1e9))
scores.shape, attn_unmasked.shape

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9, 4))
ax[0].imshow(attn_unmasked, cmap='viridis'); ax[0].set_title('unmasked')
ax[1].imshow(attn_causal,   cmap='viridis'); ax[1].set_title('causal mask')
for a in ax:
    a.set_xlabel('key'); a.set_ylabel('query')
    a.set_xticks(range(T)); a.set_yticks(range(T))
plt.tight_layout(); plt.show()

## What to look for

- The unmasked matrix has positive weights everywhere — every token can
  attend to every other token.
- The causal matrix is strictly lower-triangular: token `i` only attends to
  positions `≤ i`. This is what makes a *decoder* (and a GPT) generate text
  one token at a time without cheating.

## Next step

[03_block_shapes.ipynb](./03_block_shapes.ipynb) — forward a real tensor
through a full transformer block and check shapes.